# ============================================================
# PAMAP2 Human Activity Recognition
# Deep Learning Data Preparation
# Sliding Window Generation
# ============================================================
#
# Dataset: PAMAP2
# Purpose:
#   Prepare fixed-length sliding windows from the preprocessed
#   PAMAP2 dataset for deep learning models (TinyHAR, CNN, LSTM).
#
# ============================================================

1. Import Libraries

In [17]:
import numpy as np
import pandas as pd

from tqdm import tqdm

In [8]:
import os
os.environ.pop("PIP_PREFIX", None)

import sys
!{sys.executable} -m pip install --no-index --user --ignore-installed tqdm

Looking in links: /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v4, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/x86-64-v3, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/gentoo2023/generic, /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic
Processing /cvmfs/soft.computecanada.ca/custom/python/wheelhouse/generic/tqdm-4.68.4+computecanada-py3-none-any.whl


2. Load Preprocessed Dataset

In [32]:
DATA_PATH = "/lustre09/project/6081099/reem2005/DATASET/pamap2_preprocessed.csv"

data = pd.read_csv(DATA_PATH)

print("="*60)
print("Dataset Successfully Loaded")
print("="*60)
print(f"Shape : {data.shape}")
print(f"Columns : {len(data.columns)}")

Dataset Successfully Loaded
Shape : (1936481, 42)
Columns : 42


In [20]:
common_activities = None

for subject in data["subject"].unique():
    acts = set(data[data["subject"] == subject]["activityID"].unique())

    if common_activities is None:
        common_activities = acts
    else:
        common_activities &= acts

print("Common activities:", sorted(common_activities))

Common activities: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(12), np.int64(13), np.int64(16), np.int64(17)]


In [33]:
# ============================================================
# Keep only common activities
# ============================================================

common_activities = [1, 2, 3, 4, 12, 13, 16, 17]

data = data[data["activityID"].isin(common_activities)].copy()

print(data["activityID"].unique())

[ 1  2  3 17 16 12 13  4]


In [34]:
label_mapping = {
    1: 0,
    2: 1,
    3: 2,
    4: 3,
    12: 4,
    13: 5,
    16: 6,
    17: 7
}

data["activityID"] = data["activityID"].map(label_mapping)
print(sorted(data["activityID"].unique()))

[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]


In [35]:
# ============================================================
# Sliding Window Parameters
# ============================================================

WINDOW_SIZE = 100      # 2 seconds @ 50Hz
STEP_SIZE = 50         # 50% overlap

In [36]:
import numpy as np

def create_windows(df, window_size=100, step_size=50):

    windows = []
    labels = []
    subjects = []

    sensor_columns = [
        "hand_acc16_x","hand_acc16_y","hand_acc16_z",
        "hand_gyro_x","hand_gyro_y","hand_gyro_z",

        "chest_acc16_x","chest_acc16_y","chest_acc16_z",
        "chest_gyro_x","chest_gyro_y","chest_gyro_z",

        "ankle_acc16_x","ankle_acc16_y","ankle_acc16_z",
        "ankle_gyro_x","ankle_gyro_y","ankle_gyro_z",
    ]

    start = 0
    n = len(df)

    while start + window_size <= n:

        end = start + window_size

        window = df.iloc[start:end]

        if window["activityID"].nunique() == 1:

            windows.append(
                window[sensor_columns].to_numpy(dtype=np.float32)
            )

            labels.append(
                window["activityID"].iloc[0]
            )

            subjects.append(
                window["subject"].iloc[0]
            )

            start += step_size

        else:

            current_activity = df.iloc[start]["activityID"]

            while start + 1 < n and df.iloc[start + 1]["activityID"] == current_activity:
                start += 1

            start += 1

    return windows, labels, subjects

In [37]:
# ============================================================
# Create Windows
# ============================================================

X = []
y = []
groups = []

for subject, subject_df in data.groupby("subject"):

    subject_df = subject_df.sort_values("timestamp")

    w, l, g = create_windows(
        subject_df,
        WINDOW_SIZE,
        STEP_SIZE
    )

    X.extend(w)
    y.extend(l)
    groups.extend(g)

X = np.array(X, dtype=np.float32)
y = np.array(y)
groups = np.array(groups)

print("X:", X.shape)
print("y:", y.shape)
print("groups:", groups.shape)

X: (28727, 100, 18)
y: (28727,)
groups: (28727,)


In [38]:
sensor_columns = [

    column
    for column in data.columns
    if column not in [
        "timestamp",
        "activityID",
        "subject"
    ]
]

print("="*60)
print("Sensor Configuration")
print("="*60)
print(f"Sensor Channels : {len(sensor_columns)}")

Sensor Configuration
Sensor Channels : 39


In [40]:
# ============================================================
# Dataset Summary
# ============================================================

print("=" * 60)
print("Sliding Window Generation Complete")
print("=" * 60)

print(f"Number of windows : {len(X):,}")
print(f"Window size       : {WINDOW_SIZE}")
print(f"Sensor channels   : {X.shape[2]}")
print(f"X shape           : {X.shape}")
print(f"y shape           : {y.shape}")
print(f"Subjects          : {len(np.unique(groups))}")

Sliding Window Generation Complete
Number of windows : 28,727
Window size       : 100
Sensor channels   : 18
X shape           : (28727, 100, 18)
y shape           : (28727,)
Subjects          : 8


In [42]:
# ============================================================
# Label Distribution
# ============================================================

label_distribution = pd.Series(y).value_counts().sort_index()

distribution = pd.DataFrame({
    "Activity": label_distribution.index,
    "Windows": label_distribution.values
})

display(distribution)

,Activity,Windows
0,0,3837
1,1,3691
2,2,3788
3,3,4763
4,4,2322
5,5,2072
6,6,3494
7,7,4760


In [44]:
# ============================================================
# Save Windowed Dataset
# ============================================================

SAVE_PATH = "/lustre09/project/6081099/reem2005/DATASET/"

np.save(f"{SAVE_PATH}/X_windows.npy", X)
np.save(f"{SAVE_PATH}/y_windows.npy", y)
np.save(f"{SAVE_PATH}/groups.npy", groups)
print("Windowed dataset saved successfully.")

Windowed dataset saved successfully.
